In [ ]:
# ===== CONFIG =====
import os
INPUT_DIR   = os.environ.get("ITDA_INPUT_DIR",  "./val_images")
OUTPUT_PATH = os.environ.get("ITDA_OUTPUT_PATH", "./submission.csv")
# ==================


In [ ]:
# ===== 환경 설정 =====
# 채점 환경: Standard 4-Core vCPU, GPU 없음, 오프라인, Python 3.10
import os, sys, glob, time
os.environ.setdefault("OMP_NUM_THREADS", "4")

import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageOps

# 이 노트북 폴더를 기준으로 잡음. 저장소 루트에서 실행하든 이 폴더에서
# 실행하든 둘 다 되게 하려고 config_ko.yaml이 있는 쪽을 찾아서 씀.
BASE = os.getcwd()
if not os.path.exists(os.path.join(BASE, "config_ko.yaml")):
    _cand = os.path.join(BASE, "alt_pipeline_rapidocr")
    if os.path.exists(os.path.join(_cand, "config_ko.yaml")):
        BASE = _cand
sys.path.insert(0, BASE)

# date_parser_plus는 저장소 루트의 date_parser.py를 가져다 쓰고,
# 그게 실패한 건에만 보강 단계를 더 태우는 층임.
from date_parser_plus import extract_expiry_fields
from preprocess import enhance                   # 대비 보정 + 끊긴 글자 이어붙이기

# OCR — 가중치는 저장소에 동봉되어 오프라인에서도 동작
from rapidocr_onnxruntime import RapidOCR

def _make_cfg():
    """config에 적힌 상대경로를 rapidocr가 자기 패키지 위치 기준으로
    해석해버려서 파일을 못 찾음. 런타임에 절대경로로 바꿔친 임시 config를 만듦."""
    src_cfg = os.path.join(BASE, "config_ko.yaml")
    if not os.path.exists(src_cfg):
        return None
    import tempfile, re
    txt = open(src_cfg, encoding="utf-8").read()
    def _abs(m):
        rel = m.group(1).strip()
        cand = os.path.join(BASE, rel.lstrip("./"))
        return f"model_path: {cand}" if os.path.exists(cand) else m.group(0)
    txt = re.sub(r"model_path:\s*(\./[^\n]+)", _abs, txt)
    fd, out = tempfile.mkstemp(suffix=".yaml"); os.close(fd)
    open(out, "w", encoding="utf-8").write(txt)
    return out

RAPID_CFG = _make_cfg()
ocr = RapidOCR(config_path=RAPID_CFG) if RAPID_CFG else RapidOCR()

# YOLO 영역 검출기 (가중치 있을 때만 사용, 없으면 OCR 단독으로 동작)
YOLO_W = os.path.join(BASE, "weights", "region_best.pt")
det = None
if os.path.exists(YOLO_W):
    try:
        from ultralytics import YOLO
        det = YOLO(YOLO_W)
        print("YOLO 검출기 로드 완료")
    except Exception as e:
        print(f"YOLO 로드 실패, OCR 단독으로 진행: {e}")
else:
    print("YOLO 가중치 없음 → OCR 단독 모드")

MAX_SIDE = 1600
DET_IMGSZ = 960
CLS = {0: "date", 1: "due", 2: "code", 3: "full"}
print(f"입력: {INPUT_DIR}")


In [ ]:
# ===== 유틸 =====
def load_bgr(path):
    """EXIF 회전 적용. 학습셋 정규화와 동일 규약 (전체의 8.7%가 세로 촬영)."""
    try:
        with Image.open(path) as im:
            im = ImageOps.exif_transpose(im).convert("RGB")
            return cv2.cvtColor(np.asarray(im), cv2.COLOR_RGB2BGR)
    except Exception:
        return None

def resize_long(img, m=MAX_SIDE):
    h, w = img.shape[:2]
    s = m / max(h, w)
    return cv2.resize(img, (round(w*s), round(h*s)), interpolation=cv2.INTER_AREA) if s < 1 else img

def read_text(img):
    try:
        r, _ = ocr(img)
        return " ".join(x[1] for x in (r or []))
    except Exception:
        return ""

def pad_clip(b, W, H, pad=0.30, minpx=12):
    x0, y0, x1, y1 = b
    pw = max((x1-x0)*pad, minpx); ph = max((y1-y0)*pad, minpx)
    return (max(0,int(x0-pw)), max(0,int(y0-ph)), min(W,int(x1+pw)), min(H,int(y1+ph)))

def upscale_small(c, th=64):
    h, w = c.shape[:2]
    if h == 0 or h >= th: return c
    s = th/h
    return cv2.resize(c, (int(w*s), int(h*s)), interpolation=cv2.INTER_CUBIC)

def get_crops(img):
    """YOLO로 관심영역 크롭. 검출기 없거나 실패하면 빈 리스트."""
    if det is None: return []
    H, W = img.shape[:2]
    try:
        r = det.predict(img, imgsz=DET_IMGSZ, conf=0.25, device="cpu", verbose=False)[0]
        if r.boxes is None or len(r.boxes) == 0: return []
        g = {v: [] for v in CLS.values()}
        for b, c, cf in zip(r.boxes.xyxy.cpu().numpy(),
                            r.boxes.cls.cpu().numpy().astype(int),
                            r.boxes.conf.cpu().numpy()):
            g[CLS.get(int(c), "?")].append((b.tolist(), float(cf)))
        if g["full"]:
            boxes = [b for b, _ in sorted(g["full"], key=lambda t: -t[1])[:2]]
        elif g["date"] or g["due"]:
            a = np.asarray([b for b, _ in g["date"] + g["due"]], dtype=float)
            boxes = [[a[:,0].min(), a[:,1].min(), a[:,2].max(), a[:,3].max()]]
        else:
            return []
        out = []
        for b in boxes:
            x0, y0, x1, y1 = pad_clip(b, W, H)
            c = img[y0:y1, x0:x1]
            if c.size: out.append(upscale_small(c))
        return out
    except Exception:
        return []

NONE4 = {"year":"NONE","month":"NONE","day":"NONE","final_date":"NONE"}

def predict_one(path):
    """4단계 폴백: 크롭 → 크롭+전처리 → 전체 → 전체+전처리"""
    img = load_bgr(path)
    if img is None: return dict(NONE4)

    crops = get_crops(img)
    if crops:
        p = extract_expiry_fields(" ".join(read_text(c) for c in crops))
        if p["final_date"] != "NONE": return p
        p = extract_expiry_fields(" ".join(read_text(enhance(c)) for c in crops))
        if p["final_date"] != "NONE": return p

    small = resize_long(img)
    p = extract_expiry_fields(read_text(small))
    if p["final_date"] != "NONE": return p
    p = extract_expiry_fields(read_text(enhance(small)))
    return p if p["final_date"] != "NONE" else dict(NONE4)


In [ ]:
# ===== 추론 실행 =====
EXTS = ("*.jpg","*.jpeg","*.png","*.JPG","*.JPEG","*.PNG","*.bmp","*.webp")
paths = sorted({p for e in EXTS for p in glob.glob(os.path.join(INPUT_DIR, e))})
if not paths:
    paths = sorted(glob.glob(os.path.join(INPUT_DIR, "*.*")))
print(f"입력 이미지 {len(paths)}장")

rows = []
t0 = time.time()
for i, p in enumerate(paths, 1):
    try:
        r = predict_one(p)
    except Exception as e:
        print(f"[WARN] {os.path.basename(p)} 처리 중 오류 → NONE: {e}")
        r = dict(NONE4)
    rows.append({
        "image_id": os.path.splitext(os.path.basename(p))[0],
        "year": r["year"], "month": r["month"], "day": r["day"],
        "final_date": r["final_date"],
    })
    if i % 50 == 0 or i == len(paths):
        el = time.time() - t0
        print(f"  [{i}/{len(paths)}] {el:.1f}초 (장당 {el/i:.3f}초)")

df = pd.DataFrame(rows, columns=["image_id","year","month","day","final_date"])
df.to_csv(OUTPUT_PATH, index=False)
el = time.time() - t0
print(f"\n총 {len(df)}장 / {el:.1f}초 / 장당 {el/max(1,len(df)):.3f}초")
print(f"응답률 {(df.final_date != 'NONE').mean()*100:.1f}%")
print(f"Saved {len(df)} rows to {OUTPUT_PATH}")
